Permutation test for the VRP timing signal: shuffle the `position` series across dates thousands of times and rebuild the null Sharpe distribution. If the actual (correctly-dated) Sharpe sits in the tail of that distribution, the regime + VRP-rank timing model is adding skill on top of a random reshuffle of the same position sizes, not just riding average short-vol exposure.

In [1]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT))
from src.vrp_strategy.backtest import raw_strategy_returns, vol_target_scale
from src.vrp_strategy.metrics import performance_metrics, print_metrics

DATA_PROCESSED = ROOT / "data" / "processed"

data = pd.read_csv(DATA_PROCESSED / "positions.csv",
                   index_col=0, parse_dates=True)
SPLIT = "2016-01-01"
data = data[data.index >= SPLIT]

print(f"Out-of-sample period: {data.index[0].date()} → {data.index[-1].date()}")
print(f"Rows: {len(data)}")

Out-of-sample period: 2016-01-04 → 2026-05-22
Rows: 2612


In [2]:
# Same vol-targeting as 05_backtest.ipynb, so the test runs on the leverage-scaled
# Sharpe (the strategy's headline number), not the raw unit-exposure one.
TARGET_VOL = 0.10
VOL_WINDOW = 126
SCALAR_CAP = 100.0

def sharpe_from_position(position_values):
    position_series = pd.Series(position_values, index=data.index)
    raw_ret = raw_strategy_returns(position_series, data["iv_daily"], data["rv_daily"])
    scaled_ret, _ = vol_target_scale(raw_ret, TARGET_VOL, VOL_WINDOW, SCALAR_CAP)
    return performance_metrics(scaled_ret.dropna())["sharpe"]

actual_sharpe = sharpe_from_position(data["position"].values)
print(f"Actual (correctly-dated) Sharpe: {actual_sharpe:.3f}")

Actual (correctly-dated) Sharpe: 3.080


In [3]:
# Null distribution: 2,000 random reshuffles of the position series.
# Same values, scrambled onto different dates, so the marginal distribution
# of position sizes is unchanged, only the date-to-size mapping is destroyed.
np.random.seed(42)
N_PERM = 2000
position_arr = data["position"].values

null_sharpes = np.empty(N_PERM)
for i in range(N_PERM):
    shuffled = np.random.permutation(position_arr)
    null_sharpes[i] = sharpe_from_position(shuffled)

p_value = (1 + np.sum(null_sharpes >= actual_sharpe)) / (N_PERM + 1)

print(f"Null distribution mean:      {null_sharpes.mean():.3f}")
print(f"Null distribution std:       {null_sharpes.std():.3f}")
print(f"Null distribution 95th pct:  {np.percentile(null_sharpes, 95):.3f}")
print(f"Null distribution 99th pct:  {np.percentile(null_sharpes, 99):.3f}")
print(f"Permutations >= actual Sharpe:  {int(np.sum(null_sharpes >= actual_sharpe))} / {N_PERM}")
print(f"p-value:                       {p_value:.4f}")

Null distribution mean:      1.754
Null distribution std:       0.384
Null distribution 95th pct:  2.413
Null distribution 99th pct:  2.752
Permutations >= actual Sharpe:  7 / 2000
p-value:                       0.0040


The null distribution isn't centered on zero, and that makes sense: even a random reshuffle of which days get big vs. small positions still inherits VRP's structurally positive average return, since selling variance has earned a persistent premium over this sample. A randomly-timed strategy already comes out ahead most of the time. That's the honest baseline here, not "no strategy at all."

What the test actually isolates is narrower: does the specific regime + VRP-percentile-rank mapping from date to position size add skill on top of that baseline? p < 0.05 means fewer than 5% of random relabelings matched or beat the real result, so the timing itself, not just the exposure, is doing something.

In [4]:
pd.Series(null_sharpes, name="null_sharpe").to_csv(
    DATA_PROCESSED / "permutation_test_null_sharpes.csv", index=False)

verdict = "rejects" if p_value < 0.05 else "fails to reject"
print(f"Null hypothesis (position timing has no skill beyond a random reshuffle): {verdict} at the 5% level")
print(f"Actual Sharpe {actual_sharpe:.3f} vs null 95th pct {np.percentile(null_sharpes, 95):.3f}, p = {p_value:.4f}")
print("Saved permutation_test_null_sharpes.csv")

Null hypothesis (position timing has no skill beyond a random reshuffle): rejects at the 5% level
Actual Sharpe 3.080 vs null 95th pct 2.413, p = 0.0040
Saved permutation_test_null_sharpes.csv


In [5]:
fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(null_sharpes, bins=50, color="#7c3aed", alpha=0.6, label="Null distribution (shuffled position)")
ax.axvline(actual_sharpe, color="#dc2626", linewidth=2, label=f"Actual Sharpe = {actual_sharpe:.3f}")
ax.axvline(np.percentile(null_sharpes, 95), color="#6b7280", linestyle="--", linewidth=1, label="Null 95th percentile")
ax.set_xlabel("Vol-targeted Sharpe ratio")
ax.set_ylabel("Frequency")
ax.set_title(f"Permutation test ({N_PERM} random position reshuffles, p = {p_value:.4f})")
ax.legend(loc="upper left", fontsize=9)
plt.tight_layout()
plt.savefig(DATA_PROCESSED / "permutation_test.png", dpi=150, bbox_inches="tight")
plt.show()